# Comparing the voting records of Diane Abbott and Jack Abbott

This notebook collects the House of Commons voting records of Diane Abbott (member ID 172) and Jack Abbott (member ID 5131) and compares how they voted. It first scrapes Diane Abbott's voting pages on the UK Parliament website with requests and BeautifulSoup. When the site started blocking requests, it switches to Parliament's Commons Votes API, which returns every vote with its date as JSON. Pandas then joins the two records into one table, matching votes on bill title and date.

It needs Python with requests, beautifulsoup4 and pandas. The main outputs are abbott_votes_combined.csv, with one row per MP per vote, and abbott_votes_side_by_side.csv, with one row per vote and a column for each MP, where "No vote recorded" means that MP did not take part. The raw API data is saved as diane_abbott_api.json and jack_abbott_api.json, and the first scrape, which has no dates, as diane_abbott_votes.json and diane_abbott_votes.csv. The records go back to March 2016, and in the saved run the two MPs took part in 300 of the same votes and voted differently in 33.

In [ ]:
import requests
from bs4 import BeautifulSoup as bs

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
}
req = requests.get(url, headers=headers)
req.status_code

200

In [ ]:
soup = bs(req.text, "html.parser")
my_votes = soup.select("a.card-member-vote .result-large .text")
print(len(my_votes))

for v in my_votes:
    print(v.get_text(strip=True))

20
No
Aye
No
No
No
No
No
No
Aye
Aye
Aye
Aye
Aye
Aye
Aye
No
No
Aye
Aye
Aye


In [ ]:
votes = {"Aye": [], "No": []}

for card in soup.select("a.card-member-vote"):
    bill = card.select_one(".primary-info").get_text(strip=True)
    vote = card.select_one(".result-large .text").get_text(strip=True)
    votes.setdefault(vote, []).append(bill)

print(len(votes["Aye"]), "Aye votes")
print(len(votes["No"]), "No votes")
votes

11 Aye votes
9 No votes


{'Aye': ['Immigration and Asylum Bill Report Stage: New Clause 62',
  'Privilege',
  'Victims and Courts Bill: motion to disagree with Lords Amendment 6',
  'Victims and Courts Bill: motion to disagree with Lords Amendment 5',
  'Victims and Courts Bill: motion to disagree with  Lords Amendment 4',
  'Victims and Courts Bill: motion to disagree with Lords Amendment 3',
  'Victims and Courts Bill: motion to disagree with Lords Amendment 2',
  'Victims and Courts Bill: motion to disagree with Lords Amendment 1',
  'National Insurance Contributions (Employer Pensions Contributions) Bill: motion to disagree with Lords Amendment 6',
  'National Insurance Contributions (Employer Pensions Contributions) Bill: motion to disagree with Lords Amendment 5',
  'National Insurance Contributions (Employer Pensions Contributions) Bill: motion to disagree with Lords Amendment 3'],
 'No': ['Immigration and Asylum Bill: Third Reading',
  'Immigration and Asylum Bill Report Stage: New Clause 54',
  'Immig

In [ ]:
import re
import time

member_id = 172
url = f"https://members.parliament.uk/member/{member_id}/voting"
votes = {"name": "", "member_id": member_id, "Aye": [], "No": []}

for page in range(1, 101):
    req = requests.get(url, params={"page": page}, headers=headers)
    if req.status_code != 200:
        print("Stopped at page", page, "with status", req.status_code)
        break

    soup = bs(req.text, "html.parser")

    if page == 1:
        title = soup.title.get_text(strip=True)
        votes["name"] = re.split(r" [-–] ", title.replace("Voting record for ", ""))[0]

    cards = soup.select("a.card-member-vote")
    if not cards:
        break

    for card in cards:
        bill = card.select_one(".primary-info").get_text(strip=True)
        vote = card.select_one(".result-large .text").get_text(strip=True)
        votes.setdefault(vote, []).append(bill)

    print("Page", page, "done")
    time.sleep(1)

print(votes["name"])
print(len(votes["Aye"]), "Aye votes")
print(len(votes["No"]), "No votes")

Page 1 done
Page 2 done
Page 3 done
Page 4 done
Page 5 done
Page 6 done
Page 7 done
Page 8 done
Page 9 done
Page 10 done
Page 11 done
Page 12 done
Page 13 done
Page 14 done
Page 15 done
Page 16 done
Page 17 done
Page 18 done
Page 19 done
Page 20 done
Page 21 done
Page 22 done
Page 23 done
Page 24 done
Page 25 done
Page 26 done
Page 27 done
Page 28 done
Page 29 done
Page 30 done
Page 31 done
Page 32 done
Page 33 done
Page 34 done
Page 35 done
Page 36 done
Page 37 done
Page 38 done
Page 39 done
Page 40 done
Page 41 done
Page 42 done
Page 43 done
Page 44 done
Page 45 done
Page 46 done
Page 47 done
Page 48 done
Page 49 done
Page 50 done
Page 51 done
Page 52 done
Page 53 done
Page 54 done
Page 55 done
Page 56 done
Page 57 done
Page 58 done
Page 59 done
Page 60 done
Page 61 done
Page 62 done
Page 63 done
Page 64 done
Page 65 done
Page 66 done
Page 67 done
Page 68 done
Page 69 done
Page 70 done
Page 71 done
Page 72 done
Page 73 done
Page 74 done
Page 75 done
Page 76 done
Page 77 done
Ms Diane

In [ ]:
import json

with open("diane_abbott_votes.json", "w") as f:
    json.dump(votes, f, indent=2)

In [ ]:
import pandas as pd

rows = [{"name": votes["name"], "vote": v, "bill": b} for v in ["Aye", "No"] for b in votes[v]]
df = pd.DataFrame(rows)
df.to_csv("diane_abbott_votes.csv", index=False)

In [ ]:
{"Closure motion": [{"date": "11 September 2026", "vote": "No"},
                    {"date": "...", "vote": "Aye"}]}

{'Closure motion': [{'date': '11 September 2026', 'vote': 'No'},
  {'date': '...', 'vote': 'Aye'}]}

In [ ]:
import re
import time

member_id = 172
url = f"https://members.parliament.uk/member/{member_id}/voting"
votes = {"name": "", "member_id": member_id, "bills": {}}

for page in range(1, 101):
    req = requests.get(url, params={"page": page}, headers=headers)
    if req.status_code != 200:
        print("Stopped at page", page, "with status", req.status_code)
        break

    soup = bs(req.text, "html.parser")

    if page == 1:
        title = soup.title.get_text(strip=True)
        votes["name"] = re.split(r" [-–] ", title.replace("Voting record for ", ""))[0]

    cards = soup.select("a.card-member-vote")
    if not cards:
        break

    for card in cards:
        bill = card.select_one(".primary-info").get_text(strip=True)
        vote = card.select_one(".result-large .text").get_text(strip=True)
        found = re.search(r"held on (\d{1,2} \w+ \d{4})", card.get_text(" ", strip=True))
        date = found.group(1) if found else None
        votes["bills"].setdefault(bill, []).append({"date": date, "vote": vote})

    print("Page", page, "done")
    time.sleep(1)

total = sum(len(v) for v in votes["bills"].values())
print(votes["name"])
print(len(votes["bills"]), "different bills,", total, "votes")

Page 1 done
Page 2 done
Page 3 done
Page 4 done
Stopped at page 5 with status 403
Ms Diane Abbott
80 different bills, 80 votes


In [ ]:
list(votes["bills"].items())[:3]

[('Immigration and Asylum Bill: Third Reading',
  [{'date': '17 November 2026', 'vote': 'No'}]),
 ('Immigration and Asylum Bill Report Stage: New Clause 62',
  [{'date': '17 November 2026', 'vote': 'Aye'}]),
 ('Immigration and Asylum Bill Report Stage: New Clause 54',
  [{'date': '17 November 2026', 'vote': 'No'}])]

In [ ]:
api = "https://commonsvotes-api.parliament.uk/data/divisions.json/membervoting"
votes_api = {"name": votes["name"], "member_id": member_id, "bills": {}}
skip = 0

while True:
    params = {
        "queryParameters.memberId": member_id,
        "queryParameters.skip": skip,
        "queryParameters.take": 25,
    }
    data = requests.get(api, params=params, headers=headers).json()
    if not data:
        break

    for item in data:
        division = item["PublishedDivision"]
        bill = division["Title"]
        date = division["Date"][:10]
        if item["MemberVotedAye"]:
            vote = "Aye"
        elif item["MemberVotedNo"]:
            vote = "No"
        else:
            vote = "Teller"
        votes_api["bills"].setdefault(bill, []).append({"date": date, "vote": vote})

    skip += 25
    print(skip, "votes fetched")
    time.sleep(0.5)

total = sum(len(v) for v in votes_api["bills"].values())
print(len(votes_api["bills"]), "different bills,", total, "votes")

25 votes fetched
50 votes fetched
75 votes fetched
100 votes fetched
125 votes fetched
150 votes fetched
175 votes fetched
200 votes fetched
225 votes fetched
250 votes fetched
275 votes fetched
300 votes fetched
325 votes fetched
350 votes fetched
375 votes fetched
400 votes fetched
425 votes fetched
450 votes fetched
475 votes fetched
500 votes fetched
525 votes fetched
550 votes fetched
575 votes fetched
600 votes fetched
625 votes fetched
650 votes fetched
675 votes fetched
700 votes fetched
725 votes fetched
750 votes fetched
775 votes fetched
800 votes fetched
825 votes fetched
850 votes fetched
875 votes fetched
900 votes fetched
925 votes fetched
950 votes fetched
975 votes fetched
1000 votes fetched
1025 votes fetched
1050 votes fetched
1075 votes fetched
1100 votes fetched
1125 votes fetched
1150 votes fetched
1175 votes fetched
1200 votes fetched
1225 votes fetched
1250 votes fetched
1275 votes fetched
1300 votes fetched
1325 votes fetched
1350 votes fetched
1375 votes fetch

In [ ]:
import pandas as pd

rows = []
for bill, entries in votes_api["bills"].items():
    for entry in entries:
        rows.append({
            "name": votes_api["name"],
            "member_id": votes_api["member_id"],
            "bill": bill,
            "date": entry["date"],
            "vote": entry["vote"],
        })

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["bill", "date"])

print(len(df), "rows")
df.head(10)

1527 rows


,name,member_id,bill,date,vote
153,Ms Diane Abbott,172,\tEmployment Rights Bill: Motion to disagree w...,2025-09-15,Aye
149,Ms Diane Abbott,172,\tEmployment Rights Bill: Motion to disagree w...,2025-09-15,Aye
261,Ms Diane Abbott,172,\tNational Insurance Contributions (Secondary ...,2025-03-19,Aye
799,Ms Diane Abbott,172,Coronavirus Act 2020 (Review of Temporary Pro...,2021-03-25,No
26,Ms Diane Abbott,172,Draft Employment Rights Act 2025 (Investigato...,2026-03-18,No
156,Ms Diane Abbott,172,Employment Rights Bill: Motion to disagree wi...,2025-09-15,Aye
1471,Ms Diane Abbott,172,Finance Bill: committee: New Clause 2,2016-06-28,Aye
684,Ms Diane Abbott,172,Health and Care Bill Report Stage: New Clause 12,2021-11-23,Aye
952,Ms Diane Abbott,172,Abortion (Northern Ireland) (No. 2) Regulation...,2020-06-17,Aye
767,Ms Diane Abbott,172,Abortion (Northern Ireland) Regulations 2021,2021-04-27,Aye


In [ ]:
import json

with open("jack_abbott_api.json", "w") as f:
    json.dump(votes_api, f, indent=2)

In [ ]:
import json
import pandas as pd

with open("jack_abbott_api.json") as f:
    jack_api = json.load(f)

def to_table(data):
    rows = []
    for bill, entries in data["bills"].items():
        for entry in entries:
            rows.append({
                "name": data["name"],
                "member_id": data["member_id"],
                "bill": bill,
                "date": entry["date"],
                "vote": entry["vote"],
            })
    table = pd.DataFrame(rows)
    table["date"] = pd.to_datetime(table["date"])
    table["bill"] = table["bill"].str.replace(r"\s+", " ", regex=True).str.strip()
    return table

both = pd.concat([to_table(votes_api), to_table(jack_api)], ignore_index=True)
both = both.sort_values(["bill", "date", "name"])

print(both["name"].value_counts())
both.head(10)

name
Ms Diane Abbott    3054
Name: count, dtype: int64


,name,member_id,bill,date,vote
952,Ms Diane Abbott,172,Abortion (Northern Ireland) (No. 2) Regulation...,2020-06-17,Aye
2479,Ms Diane Abbott,172,Abortion (Northern Ireland) (No. 2) Regulation...,2020-06-17,Aye
767,Ms Diane Abbott,172,Abortion (Northern Ireland) Regulations 2021,2021-04-27,Aye
2294,Ms Diane Abbott,172,Abortion (Northern Ireland) Regulations 2021,2021-04-27,Aye
734,Ms Diane Abbott,172,"Adjournment (Conference, November and Christmas)",2021-06-23,Aye
2261,Ms Diane Abbott,172,"Adjournment (Conference, November and Christmas)",2021-06-23,Aye
443,Ms Diane Abbott,172,"Adjournment (Summer, Conference and Christmas ...",2023-07-12,Aye
1970,Ms Diane Abbott,172,"Adjournment (Summer, Conference and Christmas ...",2023-07-12,Aye
746,Ms Diane Abbott,172,Advanced Research and Invention Agency Bill: R...,2021-06-07,Aye
2273,Ms Diane Abbott,172,Advanced Research and Invention Agency Bill: R...,2021-06-07,Aye


In [ ]:
import json
import time
import requests
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}
api = "https://commonsvotes-api.parliament.uk/data/divisions.json/membervoting"

def fetch_votes(member_id, name):
    result = {"name": name, "member_id": member_id, "bills": {}}
    skip = 0
    while True:
        params = {
            "queryParameters.memberId": member_id,
            "queryParameters.skip": skip,
            "queryParameters.take": 25,
        }
        data = requests.get(api, params=params, headers=headers).json()
        if not data:
            break
        for item in data:
            division = item["PublishedDivision"]
            if item["MemberVotedAye"]:
                vote = "Aye"
            elif item["MemberVotedNo"]:
                vote = "No"
            else:
                vote = "Teller"
            result["bills"].setdefault(division["Title"], []).append(
                {"date": division["Date"][:10], "vote": vote}
            )
        skip += 25
        print(name, skip, "votes fetched")
        time.sleep(0.5)
    return result

def to_table(data):
    rows = []
    for bill, entries in data["bills"].items():
        for entry in entries:
            rows.append({
                "name": data["name"],
                "member_id": data["member_id"],
                "bill": bill,
                "date": entry["date"],
                "vote": entry["vote"],
            })
    table = pd.DataFrame(rows)
    table["date"] = pd.to_datetime(table["date"])
    table["bill"] = table["bill"].str.replace(r"\s+", " ", regex=True).str.strip()
    return table

existing = globals().get("votes_api", {})
if existing.get("member_id") == 172:
    diane_api = existing
else:
    diane_api = fetch_votes(172, "Ms Diane Abbott")

jack_api = fetch_votes(5131, "Jack Abbott")

with open("diane_abbott_api.json", "w") as f:
    json.dump(diane_api, f, indent=2)
with open("jack_abbott_api.json", "w") as f:
    json.dump(jack_api, f, indent=2)

both = pd.concat([to_table(diane_api), to_table(jack_api)], ignore_index=True)
both = both.sort_values(["bill", "date", "name"])

print(both["name"].value_counts())
both.head(10)

Jack Abbott 25 votes fetched
Jack Abbott 50 votes fetched
Jack Abbott 75 votes fetched
Jack Abbott 100 votes fetched
Jack Abbott 125 votes fetched
Jack Abbott 150 votes fetched
Jack Abbott 175 votes fetched
Jack Abbott 200 votes fetched
Jack Abbott 225 votes fetched
Jack Abbott 250 votes fetched
Jack Abbott 275 votes fetched
Jack Abbott 300 votes fetched
Jack Abbott 325 votes fetched
Jack Abbott 350 votes fetched
Jack Abbott 375 votes fetched
Jack Abbott 400 votes fetched
Jack Abbott 425 votes fetched
Jack Abbott 450 votes fetched
Jack Abbott 475 votes fetched
Jack Abbott 500 votes fetched
Jack Abbott 525 votes fetched
Jack Abbott 550 votes fetched
Jack Abbott 575 votes fetched
Jack Abbott 600 votes fetched
name
Ms Diane Abbott    1527
Jack Abbott         592
Name: count, dtype: int64


,name,member_id,bill,date,vote
952,Ms Diane Abbott,172,Abortion (Northern Ireland) (No. 2) Regulation...,2020-06-17,Aye
767,Ms Diane Abbott,172,Abortion (Northern Ireland) Regulations 2021,2021-04-27,Aye
734,Ms Diane Abbott,172,"Adjournment (Conference, November and Christmas)",2021-06-23,Aye
443,Ms Diane Abbott,172,"Adjournment (Summer, Conference and Christmas ...",2023-07-12,Aye
746,Ms Diane Abbott,172,Advanced Research and Invention Agency Bill: R...,2021-06-07,Aye
745,Ms Diane Abbott,172,Advanced Research and Invention Agency Bill: R...,2021-06-07,Aye
744,Ms Diane Abbott,172,Advanced Research and Invention Agency Bill: R...,2021-06-07,Aye
901,Ms Diane Abbott,172,Agriculture Bill Lords Amendments: Motion to d...,2020-10-12,No
874,Ms Diane Abbott,172,Agriculture Bill: Lords Amendment 16B Motion t...,2020-11-04,No
899,Ms Diane Abbott,172,Agriculture Bill: Lords Amendments Motion to d...,2020-10-12,No


In [ ]:
both.to_csv("abbott_votes_combined.csv", index=False)

In [ ]:
pd.set_option("display.max_rows", None)

side_by_side = both.pivot_table(index=["bill", "date"], columns="name", values="vote", aggfunc="first")
shared = side_by_side.dropna()

print(len(shared), "votes they both took part in")
shared

300 votes they both took part in


,name,Jack Abbott,Ms Diane Abbott
bill,date,,
Armed Forces Commissioner Bill Report Stage: Amendment 10,2025-01-21,No,No
Armed Forces Commissioner Bill Report Stage: Amendment 2,2025-01-21,No,No
Armed Forces Commissioner Bill Report Stage: Amendment 9,2025-01-21,No,No
Armed Forces Commissioner Bill: Motion to insist on 2A and disagree with LA2B and LA2C,2025-07-02,Aye,Aye
Biodiversity Beyond National Jurisdiction Bill: Committee: Amendment 4,2025-11-17,No,No
Biodiversity Beyond National Jurisdiction Bill: Committee: Amendment 5,2025-11-17,No,No
"Border Security, Asylum and Immigration Bill Report Stage: New Clause 14",2025-05-12,No,No
"Border Security, Asylum and Immigration Bill Report Stage: New Clause 18",2025-05-12,No,No
"Border Security, Asylum and Immigration Bill Report Stage: New Clause 3",2025-05-12,No,Aye


In [ ]:
disagree = shared[shared["Jack Abbott"] != shared["Ms Diane Abbott"]]
print(len(disagree), "votes where they voted differently")
disagree

33 votes where they voted differently


,name,Jack Abbott,Ms Diane Abbott
bill,date,,
"Border Security, Asylum and Immigration Bill Report Stage: New Clause 3",2025-05-12,No,Aye
Children's Wellbeing and Schools Bill Report Stage: New Clause 34,2025-03-18,No,Aye
Closure motion,2025-05-16,Aye,No
Courts and Tribunals Bill: Reasoned Amendment to Second Reading,2026-03-10,No,Aye
Courts and Tribunals Bill: Second Reading,2026-03-10,Aye,No
Draft Employment Rights Act 2025 (Investigatory Powers) (Consequential Amendments) Regulations 2026,2026-03-18,Aye,No
Draft Higher Education (Fee Limits and Fee Limit Condition) (England) (Amendment) Regulations 2026,2026-03-18,Aye,No
Draft Public Order Act 2023 (Interference With Use or Operation of Key National Infrastructure) Regulations 2025,2026-01-14,Aye,No
English Devolution and Community Empowerment Bill Report Stage: New Clause 29,2025-11-24,No,Aye


In [ ]:
full = both.pivot_table(index=["bill", "date"], columns="name", values="vote", aggfunc="first")
full = full.fillna("No vote recorded")

print(len(full), "votes in total")
full

1819 votes in total


,name,Jack Abbott,Ms Diane Abbott
bill,date,,
Abortion (Northern Ireland) (No. 2) Regulations 2020,2020-06-17,No vote recorded,Aye
Abortion (Northern Ireland) Regulations 2021,2021-04-27,No vote recorded,Aye
"Adjournment (Conference, November and Christmas)",2021-06-23,No vote recorded,Aye
"Adjournment (Summer, Conference and Christmas Recess)",2023-07-12,No vote recorded,Aye
Advanced Research and Invention Agency Bill: Report Stage: Amendment 1,2021-06-07,No vote recorded,Aye
Advanced Research and Invention Agency Bill: Report Stage: Amendment 12,2021-06-07,No vote recorded,Aye
Advanced Research and Invention Agency Bill: Report Stage: Amendment 14,2021-06-07,No vote recorded,Aye
Agriculture Bill Lords Amendments: Motion to disagree with LA11,2020-10-12,No vote recorded,No
Agriculture Bill: Lords Amendment 16B Motion to disagree,2020-11-04,No vote recorded,No


In [ ]:
full.to_csv("abbott_votes_side_by_side.csv")

In [ ]:
import os
print(os.path.abspath("abbott_votes_side_by_side.csv"))

/Users/sayanghosh/Documents/coding/abbott_votes_side_by_side.csv
